# Rotterdam LST, NDVI, and LCZ Rasters

Prerequsites:

```
pip install geemap ee geopandas rasterio rasterstats osmnx shapely numpy tqdm sklearn
```

## Initialise the Project

In [13]:
import os
import ee
import geemap
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import osmnx as ox
from shapely.ops import unary_union
from shapely.geometry import mapping, Point
import numpy as np
from rasterio.features import geometry_mask
from tqdm import tqdm

# ==========================================
# STEP 1: INITIALIZE EARTH ENGINE
# ==========================================

try:
    ee.Initialize(project='applied-spatial-rotterdam')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='applied-spatial-rotterdam')

# ==========================================
# STEP 2: DEFINE AOI & PROCESSING FUNCTIONS
# ==========================================

rotterdam_aoi = ee.Geometry.Rectangle([3.9315, 51.8176, 4.7, 52.0125])

def mask_landsat_clouds(image):
    """Mask clouds and cloud shadows using QA_PIXEL band (bits 3 and 4)."""
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 4).eq(0).And(qa.bitwiseAnd(1 << 3).eq(0))
    return image.updateMask(mask)

def process_landsat_indicators(image):
    """Scale optical/thermal bands and derive LST (Celsius) and NDVI."""
    optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    ndvi = optical.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    lst_celsius = (
        image.select('ST_B10')
        .multiply(0.00341802).add(149.0).subtract(273.15)
        .rename('LST_Celsius')
    )
    return (
        image
        .addBands(optical, None, True)
        .addBands(lst_celsius, None, True)
        .addBands(ndvi)
    )

def normalize_lst(image):
    """Subtract per-image spatial mean to produce LST anomaly (normalized LST)."""
    lst = image.select('LST_Celsius')
    image_mean = lst.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=rotterdam_aoi,
        scale=30,
        maxPixels=1e9
    ).getNumber('LST_Celsius')
    return (
        lst.subtract(image_mean)
        .rename('LST_Normalized')
        .toFloat()
        .copyProperties(image, image.propertyNames())
    )

## Build Landsat Composite

In [6]:
# ==========================================
# STEP 3: BUILD LANDSAT COMPOSITE
# ==========================================

landsat_collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(rotterdam_aoi)
    .filterDate('2025-05-01', '2025-08-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 30))
    .map(mask_landsat_clouds)
    .map(process_landsat_indicators)
)

print(f"Scenes found for summer 2025: {landsat_collection.size().getInfo()}")

# Normalized LST composite (anomaly relative to each image's spatial mean)
lst_normalized = (
    landsat_collection
    .map(normalize_lst)
    .select('LST_Normalized')
    .median()
    .clip(rotterdam_aoi)
)

# Median composites for NDVI and raw LST
summer_composite = landsat_collection.median().clip(rotterdam_aoi)
ndvi_layer = summer_composite.select('NDVI')

print("Finished building composite")

Scenes found for summer 2025: 6


## Extract LCZ

In [7]:
# ==========================================
# STEP 4: EXTRACT LOCAL CLIMATE ZONES (LCZ)
# ==========================================

print("Extracting Local Climate Zones...")

lcz_layer = (
    ee.ImageCollection('RUB/RUBCLIM/LCZ/global_lcz_map/latest')
    .mosaic()
    .clip(rotterdam_aoi)
    .select('LCZ_Filter')
)

print("Done")

Extracting Local Climate Zones...


## Write Rasters

In [9]:
# ==========================================
# STEP 5: EXPORT RASTERS
# ==========================================

os.makedirs('../data/rotterdam', exist_ok=True)

print("Exporting normalized LST raster (30m)...")
geemap.ee_export_image(
    lst_normalized,
    filename='../data/rotterdam/rotterdam_lst_normalized_2025.tif',
    scale=30,
    region=rotterdam_aoi,
    file_per_band=False
)

print("Exporting NDVI raster (30m)...")
geemap.ee_export_image(
    ndvi_layer,
    filename='../data/rotterdam/rotterdam_ndvi_2025.tif',
    scale=30,
    region=rotterdam_aoi,
    file_per_band=False
)

print("Exporting LCZ raster (100m)...")
geemap.ee_export_image(
    lcz_layer,
    filename='../data/rotterdam/rotterdam_lcz_2018.tif',
    scale=100,
    region=rotterdam_aoi,
    file_per_band=False
)

print("All rasters exported to ../data/rotterdam/")

Exporting normalized LST raster (30m)...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\rotterdam\rotterdam_lst_normalized_2025.tif
Exporting NDVI raster (30m)...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\rotterdam\rotterdam_ndvi_2025.tif
Exporting LCZ raster (100m)...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\rotterdam\rotterdam_lcz_2018.tif
All rasters exported to ../data/rotterdam/


# Rotterdam Rasters -> Geopackage

Note: Wijken en Buurten geopackage is to be downloaded separately, e.g. using PDOK viewer in QGIS

In [10]:
# ==========================================
# STEP 6: ZONAL STATISTICS ON GEOPACKAGE
# ==========================================

gpkg_path    = "../data/rotterdam/wijkenbuurten_2024.gpkg"
lst_raster   = "../data/rotterdam/rotterdam_lst_normalized_2025.tif"
ndvi_raster  = "../data/rotterdam/rotterdam_ndvi_2025.tif"
lcz_raster   = "../data/rotterdam/rotterdam_lcz_2018.tif"
output_gpkg  = "../data/rotterdam/rotterdam_wijkenbuurten_enriched.gpkg"

print("\nLoading GeoPackage layers...")
buurten_gdf = gpd.read_file(gpkg_path, layer="buurten")

# Keep only buurten belonging to the municipality of Rotterdam
buurten_gdf = buurten_gdf[buurten_gdf['gemeentenaam'] == 'Rotterdam'].copy()
print(f"  Filtered to {len(buurten_gdf)} buurten in Rotterdam")

# Reproject vectors to match raster CRS if needed
with rasterio.open(lst_raster) as src:
    raster_crs = src.crs

if buurten_gdf.crs != raster_crs:
    print(f"Reprojecting vectors to raster CRS: {raster_crs}")
    buurten_gdf = buurten_gdf.to_crs(raster_crs)

def extract_zonal_metrics(gdf, raster_path, stat, column_name):
    """Compute a zonal statistic from a raster for each polygon in a GeoDataFrame."""
    print(f"  {column_name} <- {stat}({os.path.basename(raster_path)})")
    stats = zonal_stats(gdf, raster_path, stats=[stat])
    gdf[column_name] = [x[stat] if x else None for x in stats]
    return gdf

print("\nProcessing Buurten (Neighborhoods)...")
buurten_gdf = extract_zonal_metrics(buurten_gdf, lst_raster,  'mean',     'mean_LST_normalized')
buurten_gdf = extract_zonal_metrics(buurten_gdf, ndvi_raster, 'mean',     'mean_NDVI')
buurten_gdf = extract_zonal_metrics(buurten_gdf, lcz_raster,  'majority', 'majority_LCZ')

print(f"\nSaving enriched layer to {output_gpkg}...")
buurten_gdf.to_file(output_gpkg, layer="buurten_enriched", driver="GPKG")

print("Done! Output files:")
print(f"  ../data/rotterdam/rotterdam_lst_normalized_2025.tif")
print(f"  ../data/rotterdam/rotterdam_ndvi_2025.tif")
print(f"  ../data/rotterdam/rotterdam_lcz_2018.tif")
print(f"  {output_gpkg}  (layer: buurten_enriched)")


Loading GeoPackage layers...
  Filtered to 92 buurten in Rotterdam
Reprojecting vectors to raster CRS: EPSG:4326

Processing Buurten (Neighborhoods)...
  mean_LST_normalized <- mean(rotterdam_lst_normalized_2025.tif)


C:\Users\artem\AppData\Local\Programs\Python\Python313\Lib\site-packages\rasterstats\io.py:437: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


  mean_NDVI <- mean(rotterdam_ndvi_2025.tif)
  majority_LCZ <- majority(rotterdam_lcz_2018.tif)

Saving enriched layer to ../data/rotterdam/rotterdam_wijkenbuurten_enriched.gpkg...
Done! Output files:
  ../data/processed/rotterdam_lst_normalized_2025.tif
  ../data/processed/rotterdam_ndvi_2025.tif
  ../data/processed/rotterdam_lcz_2018.tif
  ../data/rotterdam/rotterdam_wijkenbuurten_enriched.gpkg  (layer: buurten_enriched)


# Rotterdam Roads

## Get roads in Rotterdam

In [12]:
# Get the Rotterdam boundary
rotterdam = ox.geocode_to_gdf("Rotterdam, Netherlands")

# Reproject to Dutch RD New, buffer 1000m, reproject back
rotterdam_buffered = (
    rotterdam
    .to_crs("EPSG:28992")
    .buffer(2000)
    .to_crs("EPSG:4326")
)

# Query using the buffered polygon
G = ox.graph_from_polygon(
    unary_union(rotterdam_buffered),
    network_type="walk"
)

nodes, edges = ox.graph_to_gdfs(G)
nodes = nodes.to_crs("EPSG:28992")
edges = edges.to_crs("EPSG:28992")

# Save to file
ox.save_graphml(G, "../data/rotterdam/rotterdam_roads.graphml")

# Save manually using geopandas
nodes.to_file("../data/rotterdam/rotterdam_roads.gpkg", layer="nodes", driver="GPKG")
edges.to_file("../data/rotterdam/rotterdam_roads.gpkg", layer="edges", driver="GPKG")

# Or as GeoPackage for use in QGIS etc.
# ox.save_graph_geopackage(G, filepath="../QGIS project files/Rotterdam Network/rotterdam_roads.gpkg")

# Basic stats
print(ox.basic_stats(G))

{'n': 66851, 'm': 186772, 'k_avg': 5.587709981900046, 'edge_length_total': 9413931.24689067, 'edge_length_avg': 50.403332656343935, 'streets_per_node_avg': 2.8009304273683266, 'streets_per_node_counts': {0: 0, 1: 12328, 2: 0, 3: 43451, 4: 10814, 5: 242, 6: 15, 7: 0, 8: 1}, 'streets_per_node_proportions': {0: 0.0, 1: 0.18441010605675306, 2: 0.0, 3: 0.6499678389253714, 4: 0.16176272606243736, 5: 0.0036199907256435956, 6: 0.00022437959043245426, 7: 0.0, 8: 1.4958639362163617e-05}, 'intersection_count': 54523, 'street_length_total': 4706881.063507168, 'street_segment_count': 93382, 'street_length_avg': 50.40458614622912, 'circuity_avg': 1.0644986577897015, 'self_loop_proportion': 0.0026236319633334048}


**_IMPORTANT_**

**Before** executing next code, you should modify `rotterdam_roads` in QGIS/PST.

We recommend that you:
- Start a project with a _metric_ CRS (e.g. EPSG:28992)
- Upload the `edges` layer from `rotterdam_roads.gpkg`
- Open the processing toolbox and perform the following operations:
    - Split with lines (Input layer: `rotterdam_roads — edges`, Split layer: `rotterdam_roads — edges`)
    - Multipart to singleparts
    - If necessary, Fix geometries
- Open the PST plugin and run:
    - Create segment map
    - Perform Network Betweenness
        - Distance modes: walking distance
        - Weight modes: no weight
        - Normalisation modes: no normalisation
        - Radius: walking distance, 1000m
- Save the resulting file as `rotterdam_roads_PST.gpkg`, with the layer name `rotterdam_roads_PST`.

## Process roads with raster

In [15]:
# ── fixed paths ──────────────────────────────────────────────────────────────

VECTOR_PATH = "../data/rotterdam/rotterdam_roads_PST.gpkg"

LST_RASTER  = "../data/rotterdam/rotterdam_lst_normalized_2025.tif"
NDVI_RASTER = "../data/rotterdam/rotterdam_ndvi_2025.tif"
LCZ_RASTER  = "../data/rotterdam/rotterdam_lcz_2018.tif"

OUTPUT_PATH  = "../data/rotterdam/rotterdam_roads_data.gpkg"
OUTPUT_LAYER = "rotterdam_roads_data"

BAND = 1  # raster band used for every raster in this script


# ── helpers ──────────────────────────────────────────────────────────────────

def pixel_size(transform):
    """Return (pixel_width, pixel_height) in CRS units."""
    return abs(transform.a), abs(transform.e)


def mean_under_line(geom, src, band_index=1, buffer_fraction=0.5):
    """
    Return the mean raster value for pixels touched by `geom` (a LineString).

    Strategy
    --------
    1. Buffer the line by `buffer_fraction` x pixel diagonal so that every
       pixel the line passes through is captured.
    2. Mask the raster to that buffer window.
    3. Exclude nodata and return the mean of remaining values.
    """
    px_w, px_h = pixel_size(src.transform)
    buf_dist = buffer_fraction * (px_w**2 + px_h**2) ** 0.5

    buffered = geom.buffer(buf_dist)

    # Window covering the bounding box of the buffer
    from rasterio.windows import from_bounds
    bounds = buffered.bounds          # (minx, miny, maxx, maxy)
    window = from_bounds(*bounds, transform=src.transform).round_lengths().round_offsets()

    # Clamp window to raster extent
    window = window.intersection(
        rasterio.windows.Window(0, 0, src.width, src.height)
    )
    if window.width < 1 or window.height < 1:
        return float("nan")

    win_transform = src.window_transform(window)
    data = src.read(band_index, window=window).astype(float)

    # Mask nodata
    nodata = src.nodata
    if nodata is not None:
        data[data == nodata] = np.nan

    # Mask pixels outside the buffer polygon
    msk = geometry_mask(
        [mapping(buffered)],
        transform=win_transform,
        invert=True,           # True = inside polygon
        out_shape=data.shape,
    )
    data[~msk] = np.nan

    valid = data[~np.isnan(data)]
    return float(np.mean(valid)) if len(valid) > 0 else float("nan")


def value_at_point(point, src, band_index=1):
    """
    Return the raster value at the pixel containing `point` (a shapely Point).
    Returns NaN if the point falls outside the raster extent or on nodata.
    """
    row, col = src.index(point.x, point.y)

    if row < 0 or col < 0 or row >= src.height or col >= src.width:
        return float("nan")

    window = rasterio.windows.Window(col, row, 1, 1)
    data = src.read(band_index, window=window).astype(float)

    if data.size == 0:
        return float("nan")

    value = data[0, 0]
    if src.nodata is not None and value == src.nodata:
        return float("nan")

    return float(value)


def start_node(geom):
    """Return the first coordinate of a LineString/MultiLineString as a Point."""
    if geom.geom_type == "MultiLineString":
        first_line = list(geom.geoms)[0]
        return Point(first_line.coords[0])
    return Point(geom.coords[0])


def compute_line_mean(gdf, raster_path, band=BAND):
    """
    Compute mean_under_line for every geometry in `gdf` against the raster
    at `raster_path`, reprojecting the GeoDataFrame to the raster CRS if
    needed. Returns a list of means aligned with gdf's row order, and the
    (possibly reprojected) GeoDataFrame used for the computation.
    """
    with rasterio.open(raster_path) as src:
        print(f"  Raster CRS : {src.crs}")
        print(f"  Vector CRS : {gdf.crs}")

        work_gdf = gdf
        if gdf.crs != src.crs:
            print("  CRS mismatch — reprojecting vector to raster CRS …")
            work_gdf = gdf.to_crs(src.crs)

        means = []
        for _, row in tqdm(work_gdf.iterrows(), total=len(work_gdf),
                            desc=f"Processing lines ({raster_path.split('/')[-1]})",
                            unit="line"):
            geom = row.geometry
            if geom is None or geom.is_empty:
                means.append(float("nan"))
                continue
            if geom.geom_type == "MultiLineString":
                parts = list(geom.geoms)
                vals = [mean_under_line(p, src, band) for p in parts]
                vals = [v for v in vals if not np.isnan(v)]
                means.append(float(np.mean(vals)) if vals else float("nan"))
            else:
                means.append(mean_under_line(geom, src, band))

    return means


def compute_start_node_values(gdf, raster_path, band=BAND):
    """
    Compute value_at_point at the starting node of every geometry in `gdf`
    against the raster at `raster_path`, reprojecting if needed. Returns a
    list of values aligned with gdf's row order.
    """
    with rasterio.open(raster_path) as src:
        print(f"  Raster CRS : {src.crs}")
        print(f"  Vector CRS : {gdf.crs}")

        work_gdf = gdf
        if gdf.crs != src.crs:
            print("  CRS mismatch — reprojecting vector to raster CRS …")
            work_gdf = gdf.to_crs(src.crs)

        values = []
        for _, row in tqdm(work_gdf.iterrows(), total=len(work_gdf),
                            desc=f"Processing start nodes ({raster_path.split('/')[-1]})",
                            unit="line"):
            geom = row.geometry
            if geom is None or geom.is_empty:
                values.append(float("nan"))
                continue
            pt = start_node(geom)
            values.append(value_at_point(pt, src, band))

    return values


# ── run ──────────────────────────────────────────────────────────────────────

# ── load vector ──────────────────────────────────────────────────────────────
gdf = gpd.read_file(VECTOR_PATH)
print(f"Loaded {len(gdf)} features from '{VECTOR_PATH}'")

# ── LST: mean under line ────────────────────────────────────────────────────
print("\nComputing 'lst' (mean under line) …")
gdf["lst"] = compute_line_mean(gdf, LST_RASTER)

# ── NDVI: mean under line ───────────────────────────────────────────────────
print("\nComputing 'ndvi' (mean under line) …")
gdf["ndvi"] = compute_line_mean(gdf, NDVI_RASTER)

# ── LCZ: value at starting node only ────────────────────────────────────────
print("\nComputing 'lcz' (value at starting node) …")
gdf["lcz"] = compute_start_node_values(gdf, LCZ_RASTER)

# ── write output ─────────────────────────────────────────────────────────────
gdf.to_file(OUTPUT_PATH, driver="GPKG", layer=OUTPUT_LAYER)
print(f"\nResults written to '{OUTPUT_PATH}' (layer: '{OUTPUT_LAYER}')")

# Quick summary
for field in ("lst", "ndvi", "lcz"):
    valid = gdf[field].dropna()
    nan_count = len(gdf) - len(valid)
    print(f"\nSummary of '{field}': {len(valid)} valid, {nan_count} NaN")
    if len(valid):
        print(f"  min  : {valid.min():.4f}")
        print(f"  mean : {valid.mean():.4f}")
        print(f"  max  : {valid.max():.4f}")

Loaded 177170 features from '../data/rotterdam/rotterdam_roads_PST.gpkg'

Computing 'lst' (mean under line) …
  Raster CRS : EPSG:4326
  Vector CRS : EPSG:28992
  CRS mismatch — reprojecting vector to raster CRS …


Processing lines (rotterdam_lst_normalized_2025.tif): 100%|██████████| 177170/177170 [03:14<00:00, 911.41line/s] 



Computing 'ndvi' (mean under line) …
  Raster CRS : EPSG:4326
  Vector CRS : EPSG:28992
  CRS mismatch — reprojecting vector to raster CRS …


Processing lines (rotterdam_ndvi_2025.tif): 100%|██████████| 177170/177170 [07:56<00:00, 371.65line/s]



Computing 'lcz' (value at starting node) …
  Raster CRS : EPSG:4326
  Vector CRS : EPSG:28992
  CRS mismatch — reprojecting vector to raster CRS …


Processing start nodes (rotterdam_lcz_2018.tif): 100%|██████████| 177170/177170 [01:42<00:00, 1733.36line/s]



Results written to '../data/rotterdam/rotterdam_roads_data.gpkg' (layer: 'rotterdam_roads_data')

Summary of 'lst': 154994 valid, 22176 NaN
  min  : -8.7123
  mean : 4.1602
  max  : 20.1121

Summary of 'ndvi': 154994 valid, 22176 NaN
  min  : -0.3599
  mean : 0.4923
  max  : 0.9105

Summary of 'lcz': 177170 valid, 0 NaN
  min  : 1.0000
  mean : 6.8186
  max  : 17.0000


## Clustering

In [16]:
import sys
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.impute import SimpleImputer


# ── fixed paths / parameters ──────────────────────────────────────────────────

INPUT_PATH  = "../data/rotterdam/rotterdam_roads_data.gpkg"
LAYER_NAME  = "rotterdam_roads_data"
ATTRIBUTES  = ["Bww1kM", "lst", "ndvi"]
K           = 6
OUTPUT_PATH = "../data/rotterdam/rotterdam_roads_clustered.gpkg"
CLUSTER_COL = "cluster"
RANDOM_SEED = 42


# ── helpers ──────────────────────────────────────────────────────────────────

def load_layer(input_path, layer_name):
    import fiona
    layers = fiona.listlayers(input_path)
    if not layers:
        print(f"ERROR: No layers found in {input_path}")
        sys.exit(1)

    if layer_name is None:
        layer_name = layers[0]
        print(f"  No layer specified — using first layer: '{layer_name}'")
    elif layer_name not in layers:
        print(f"ERROR: Layer '{layer_name}' not found. Available layers: {layers}")
        sys.exit(1)

    print(f"  Loading layer '{layer_name}' from {input_path} ...")
    gdf = gpd.read_file(input_path, layer=layer_name)
    print(f"  Loaded {len(gdf):,} features.")
    return gdf, layer_name


def validate_attributes(gdf, attributes):
    missing = [a for a in attributes if a not in gdf.columns]
    if missing:
        print(f"ERROR: Attributes not found in layer: {missing}")
        print(f"  Available columns: {list(gdf.columns)}")
        sys.exit(1)

    # Check for non-numeric
    non_numeric = []
    for a in attributes:
        if not np.issubdtype(gdf[a].dtype, np.number):
            non_numeric.append(f"{a} (dtype: {gdf[a].dtype})")
    if non_numeric:
        print(f"ERROR: Non-numeric attributes detected: {non_numeric}")
        print("  All clustering attributes must be numeric.")
        sys.exit(1)


def prepare_features(gdf, attributes):
    X_raw = gdf[attributes].values

    # Report missing values
    n_missing = np.isnan(X_raw).sum()
    if n_missing > 0:
        print(f"  Warning: {n_missing:,} missing values found — imputing with column medians.")
        imputer = SimpleImputer(strategy="median")
        X_raw = imputer.fit_transform(X_raw)

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    return X_scaled


def run_clustering(X_scaled, k, random_seed, n_features):
    print(f"\n  Running K-Means with k={k}, {len(X_scaled):,} features, {n_features} attributes ...")

    # MiniBatchKMeans is much faster for large datasets (>50k rows)
    if len(X_scaled) > 50_000:
        print("  Using MiniBatchKMeans for speed (dataset > 50k rows).")
        km = MiniBatchKMeans(n_clusters=k, random_state=random_seed, n_init=10, max_iter=300)
    else:
        km = KMeans(n_clusters=k, random_state=random_seed, n_init=10, max_iter=300)

    labels = km.fit_predict(X_scaled)
    inertia = km.inertia_

    counts = np.bincount(labels)
    print(f"\n  Clustering complete. Inertia: {inertia:,.0f}")
    print(f"  Cluster sizes:")
    for i, c in enumerate(counts):
        print(f"    Cluster {i}: {c:,} features ({100*c/len(labels):.1f}%)")

    return labels


def save_output(gdf, layer_name, labels, cluster_col, output_path):
    gdf = gdf.copy()
    gdf[cluster_col] = labels.astype(int)

    print(f"\n  Writing output to {output_path} (layer: '{layer_name}') ...")
    gdf.to_file(output_path, layer=layer_name, driver="GPKG")
    print(f"  Done. New column '{cluster_col}' written with cluster labels 0–{labels.max()}.")


# ── run ──────────────────────────────────────────────────────────────────────

print("\n=== GeoPackage K-Means Clustering ===")
print(f"  Input:       {INPUT_PATH}")
print(f"  Attributes:  {ATTRIBUTES}")
print(f"  Cluster col: {CLUSTER_COL}")
print(f"  Output:      {OUTPUT_PATH}")

# Load
gdf, layer_name = load_layer(INPUT_PATH, LAYER_NAME)

# Validate
validate_attributes(gdf, ATTRIBUTES)

# Prepare feature matrix
X_scaled = prepare_features(gdf, ATTRIBUTES)

# Cluster
labels = run_clustering(X_scaled, K, RANDOM_SEED, len(ATTRIBUTES))

# Save
save_output(gdf, layer_name, labels, CLUSTER_COL, OUTPUT_PATH)
print("\n=== All done! ===\n")


=== GeoPackage K-Means Clustering ===
  Input:       ../data/rotterdam/rotterdam_roads_data.gpkg
  Attributes:  ['Bww1kM', 'lst', 'ndvi']
  Cluster col: cluster
  Output:      ../data/rotterdam/rotterdam_roads_clustered.gpkg
  Loading layer 'rotterdam_roads_data' from ../data/rotterdam/rotterdam_roads_data.gpkg ...
  Loaded 177,170 features.

  Running K-Means with k=6, 177,170 features, 3 attributes ...
  Using MiniBatchKMeans for speed (dataset > 50k rows).

  Clustering complete. Inertia: 172,665
  Cluster sizes:
    Cluster 0: 37,549 features (21.2%)
    Cluster 1: 34,707 features (19.6%)
    Cluster 2: 8,754 features (4.9%)
    Cluster 3: 26,340 features (14.9%)
    Cluster 4: 57,243 features (32.3%)
    Cluster 5: 12,577 features (7.1%)

  Writing output to ../data/rotterdam/rotterdam_roads_clustered.gpkg (layer: 'rotterdam_roads_data') ...
  Done. New column 'cluster' written with cluster labels 0–5.

=== All done! ===

